# Lab: RAG Basics - Building Retrieval Pipeline

## Learning Objectives

By the end of this lab, you will be able to:
- Understand the core components of a retrieval pipeline
- Implement document chunking strategies
- Create and work with text embeddings
- Build a vector-based retrieval system
- Evaluate retrieval quality

## Introduction

Retrieval-Augmented Generation (RAG) is a technique that enhances language models by retrieving relevant information from a knowledge base before generating responses. The retrieval pipeline is the foundation of any RAG system, responsible for finding the most relevant information given a query.


In this lab, you will build a complete retrieval pipeline from scratch, learning each component step by step.


### Required Libraries

Install the following libraries:


In [5]:

pip install sentence-transformers numpy scikit-learn

## Sample Dataset

For this lab, we'll work with a collection of documents about artificial intelligence concepts. Create a file called `documents.py` with the following content:


In [6]:
%%writefile documents.py

DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.",
    "Deep learning is a type of machine learning based on artificial neural networks with multiple layers, allowing computers to learn complex patterns.",
    "Natural language processing (NLP) is a field of AI that focuses on the interaction between computers and human language, enabling machines to understand and generate text.",
    "Computer vision is an AI field that trains computers to interpret and understand visual information from the world, such as images and videos.",
    "Reinforcement learning is a machine learning approach where an agent learns to make decisions by performing actions and receiving rewards or penalties.",
    "Transfer learning is a technique where a model developed for one task is reused as the starting point for a model on a second task.",
    "Neural networks are computing systems inspired by biological neural networks, consisting of interconnected nodes that process information.",
    "Supervised learning involves training a model on labeled data, where the correct output is known for each input example.",
    "Unsupervised learning involves training models on data without labeled responses, finding hidden patterns or structures in the data.",
    "Transformers are a type of neural network architecture that uses self-attention mechanisms, revolutionary for NLP tasks."
]

Overwriting documents.py


## Part 1: Document Preprocessing and Chunking

### Understanding the Importance

Before we can retrieve documents, we need to prepare them properly. In real-world scenarios, documents are often too large to process as single units. Chunking breaks documents into smaller, manageable pieces while preserving context.

**Why this matters:** Poor chunking can lead to lost context or irrelevant retrievals, directly impacting the quality of your RAG system.

In [7]:
### Task 1.1: Implement Basic Chunking

#Create a function that splits documents into chunks based on a maximum token/word count.
def chunk_by_words(text, max_words=50, overlap=10):
    """
    Split text into overlapping chunks based on word count.

    Args:
        text (str): Input text to chunk.
        max_words (int): Maximum number of words per chunk.
        overlap (int): Number of words shared between consecutive chunks.

    Returns:
        list: List of text chunks.
    """

    # Safety check: overlap should not be >= max_words
    if overlap >= max_words:
        raise ValueError("Overlap must be smaller than max_words")

    # Step 1: Split text into individual words
    words = text.split()

    chunks = []

    # Step size controls how far we move each time
    step = max_words - overlap

    # Step 2: Loop through words and create chunks
    for start_index in range(0, len(words), step):

        # Select words for current chunk
        end_index = start_index + max_words
        chunk_words = words[start_index:end_index]

        # Convert list of words back to string
        chunk = " ".join(chunk_words)

        chunks.append(chunk)

        # Stop if we reached the end
        if end_index >= len(words):
            break

    return chunks

In [8]:
##Test your implementation:
sample_text = "Machine learning is a subset of artificial intelligence. It enables systems to learn from experience. Deep learning uses neural networks with multiple layers."

chunks = chunk_by_words(sample_text, max_words=10, overlap=3)
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

Number of chunks: 3
Chunk 1: Machine learning is a subset of artificial intelligence. It enables
Chunk 2: intelligence. It enables systems to learn from experience. Deep learning
Chunk 3: experience. Deep learning uses neural networks with multiple layers.


In [9]:
### Task 1.2: Sentence-Based Chunking

#implement a more sophisticated chunking strategy that respects sentence boundaries.
def chunk_by_sentences(text, max_sentences=2):
    """
    Split text into chunks based on complete sentences.

    Args:
        text (str): Input text to chunk.
        max_sentences (int): Maximum number of sentences per chunk.

    Returns:
        list: List of sentence-based text chunks.
    """

    if max_sentences <= 0:
        raise ValueError("max_sentences must be greater than 0")

    # Step 1: Split text into sentences
    sentences = text.split(". ")

    # Clean and ensure proper punctuation
    cleaned_sentences = []
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence.endswith("."):
            sentence += "."
        cleaned_sentences.append(sentence)

    chunks = []

    # Step 2: Group sentences into chunks
    for i in range(0, len(cleaned_sentences), max_sentences):
        chunk_sentences = cleaned_sentences[i:i + max_sentences]
        chunk = " ".join(chunk_sentences)
        chunks.append(chunk)

    return chunks


In [10]:
sample_text = "Machine learning is powerful. It enables systems to learn from data. Deep learning uses neural networks. Transformers improve NLP."

chunks = chunk_by_sentences(sample_text, max_sentences=2)

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

Number of chunks: 2
Chunk 1: Machine learning is powerful. It enables systems to learn from data.
Chunk 2: Deep learning uses neural networks. Transformers improve NLP.


**Questions**

Q- What are the advantages of sentence-based chunking over word-based chunking?

-Sentence-based chunking keeps full sentences together, so the meaning doesn’t break. It gives better context and makes the retrieved results more clear and natural to read. It’s especially helpful in RAG systems where understanding complete ideas is important.

Q-When might word-based chunking be more appropriate?

-Word-based chunking is useful when we need strict control over size, like when there are token limits or very long documents. It helps keep all chunks uniform and manageable.

## Part 2: Creating Embeddings

### Understanding the Importance

Embeddings convert text into numerical vectors that capture semantic meaning. This transformation is crucial because computers can't directly compare text semantically, but they can measure distances between vectors.

**Why this matters:** The quality of embeddings directly determines how well your retrieval system understands semantic similarity. Poor embeddings will retrieve irrelevant documents even if your retrieval logic is perfect.

In [11]:
### Task 2.1: Initialize Embedding Model

#Load a pre-trained sentence transformer model.
from sentence_transformers import SentenceTransformer

def initialize_embedding_model(model_name='all-MiniLM-L6-v2'):
    """
    Initialize a sentence transformer model for creating embeddings.

    Args:
        model_name (str): Name of the pre-trained model.

    Returns:
        SentenceTransformer: Loaded embedding model.
    """
    try:
        model = SentenceTransformer(model_name)
        return model
    except Exception as e:
        raise RuntimeError(f"Failed to load model: {e}")


In [12]:
model = initialize_embedding_model()

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


In [13]:
### Task 2.2: Generate Document Embeddings

#Create embeddings for all documents in your knowledge base.
import numpy as np

def create_embeddings(documents, model):
    """
    Generate embeddings for a list of documents.

    Args:
        documents (list): List of text documents.
        model (SentenceTransformer): Loaded embedding model.

    Returns:
        numpy.ndarray: Array of document embeddings.
    """

    if not documents:
        raise ValueError("Document list is empty.")

    # Encode all documents at once (efficient batch processing)
    embeddings = model.encode(documents, convert_to_numpy=True)

    return embeddings

In [14]:
from documents import DOCUMENTS

model = initialize_embedding_model()
embeddings = create_embeddings(DOCUMENTS, model)

print(f"Number of documents: {len(DOCUMENTS)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"First embedding (first 10 dimensions): {embeddings[0][:10]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of documents: 10
Embedding shape: (10, 384)
First embedding (first 10 dimensions): [ 0.00840915 -0.0036058   0.05522151  0.06061025  0.03103591 -0.02438609
 -0.02974015 -0.0305842  -0.0707221  -0.00281702]


In [16]:
### Task 2.3: Understanding Embedding Space

#Calculate and visualize the similarity between different documents.
import numpy as np

def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors.

    Args:
        vec1 (numpy.ndarray): First embedding vector.
        vec2 (numpy.ndarray): Second embedding vector.

    Returns:
        float: Similarity score between -1 and 1.
    """

    # Ensure vectors are numpy arrays
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    # Compute dot product
    dot_product = np.dot(vec1, vec2)

    # Compute magnitudes
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)

    # Avoid division by zero
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0

    return dot_product / (norm_vec1 * norm_vec2)


In [17]:
#**Experiment:**


# Compare similarities between different document pairs
# Document 0: Machine learning definition
# Document 1: Deep learning definition
# Document 4: Reinforcement learning definition

similarity_ml_dl = cosine_similarity(embeddings[0], embeddings[1])
similarity_ml_rl = cosine_similarity(embeddings[0], embeddings[4])

print(f"Similarity (ML vs DL): {similarity_ml_dl:.4f}")
print(f"Similarity (ML vs RL): {similarity_ml_rl:.4f}")


Similarity (ML vs DL): 0.5919
Similarity (ML vs RL): 0.6325


**Questions:**

Q- Which pair is more similar? Why does this make sense?

-In my results, Machine Learning and Reinforcement Learning are slightly more similar.
This makes sense because reinforcement learning is described as a type of machine learning approach, so they naturally share more related concepts and terminology

Q- What does the similarity score tell you about semantic relationships?

-The similarity score shows how close two documents are in meaning.
A higher score means the documents are more semantically related, while a lower score means they are less related.

## Part 3: Building the Retrieval System

### Understanding the Importance

The retrieval system is the core of your pipeline. It takes a query and finds the most relevant documents from your knowledge base. The efficiency and accuracy of this component directly impact the performance of your entire RAG system.

**Why this matters:** A slow or inaccurate retrieval system will create bottlenecks and provide poor context to your language model, resulting in irrelevant or incorrect responses.

In [18]:
### Task 3.1: Implement Basic Retrieval

#Create a function that retrieves the top-k most similar documents for a given query.

def retrieve_documents(query, documents, embeddings, model, top_k=3):
    """
    Retrieve the most relevant documents for a query.

    Args:
        query (str): Search query string.
        documents (list): List of document texts.
        embeddings (numpy.ndarray): Pre-computed document embeddings.
        model (SentenceTransformer): Embedding model.
        top_k (int): Number of documents to retrieve.

    Returns:
        list: List of tuples (document, similarity_score).
    """

    if not query:
        raise ValueError("Query cannot be empty.")

    # Step 1: Convert query into embedding
    query_embedding = model.encode(query, convert_to_numpy=True)

    similarities = []

    # Step 2: Compute similarity with each document
    for doc, doc_embedding in zip(documents, embeddings):
        score = cosine_similarity(query_embedding, doc_embedding)
        similarities.append((doc, score))

    # Step 3: Sort by similarity (highest first)
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Step 4: Return top_k results
    return similarities[:top_k]

In [19]:
query = "How do computers learn from data?"
results = retrieve_documents(query, DOCUMENTS, embeddings, model, top_k=3)

print(f"Query: {query}\n")

for i, (doc, score) in enumerate(results, 1):
    print(f"Result {i} (Score: {score:.4f}):")
    print(f"{doc}\n")

Query: How do computers learn from data?

Result 1 (Score: 0.5245):
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.

Result 2 (Score: 0.5062):
Supervised learning involves training a model on labeled data, where the correct output is known for each input example.

Result 3 (Score: 0.4734):
Unsupervised learning involves training models on data without labeled responses, finding hidden patterns or structures in the data.



In [20]:
### Task 3.2: Implement Retrieval with Threshold

#Enhance your retrieval function to only return documents above a similarity threshold
def retrieve_with_threshold(query, documents, embeddings, model,
                            top_k=3, threshold=0.3):
    """
    Retrieve documents that meet a minimum similarity threshold.

    Args:
        query (str): Search query string.
        documents (list): List of document texts.
        embeddings (numpy.ndarray): Pre-computed document embeddings.
        model (SentenceTransformer): Embedding model.
        top_k (int): Maximum number of documents to retrieve.
        threshold (float): Minimum similarity score.

    Returns:
        list: List of tuples (document, similarity_score).
    """

    if not query:
        raise ValueError("Query cannot be empty.")

    # Step 1: Create embedding for query
    query_embedding = model.encode(query, convert_to_numpy=True)

    filtered_results = []

    # Step 2: Compute similarity and apply threshold filter
    for doc, doc_embedding in zip(documents, embeddings):
        score = cosine_similarity(query_embedding, doc_embedding)

        if score >= threshold:
            filtered_results.append((doc, score))

    # Step 3: Sort remaining results
    filtered_results.sort(key=lambda x: x[1], reverse=True)

    # Step 4: Return top_k
    return filtered_results[:top_k]

In [21]:
#**Experiment with different thresholds:**
queries = [
    "What is deep learning?",
    "How do neural networks work?",
    "Tell me about quantum computing"  # Out of domain query
]

for query in queries:
    print(f"\nQuery: {query}")
    results = retrieve_with_threshold(query, DOCUMENTS, embeddings, model,
                                      top_k=3, threshold=0.3)
    print(f"Retrieved {len(results)} documents")
    if results:
        print(f"Top result (Score: {results[0][1]:.4f}): {results[0][0][:100]}...")


Query: What is deep learning?
Retrieved 3 documents
Top result (Score: 0.7955): Deep learning is a type of machine learning based on artificial neural networks with multiple layers...

Query: How do neural networks work?
Retrieved 3 documents
Top result (Score: 0.7190): Neural networks are computing systems inspired by biological neural networks, consisting of intercon...

Query: Tell me about quantum computing
Retrieved 1 documents
Top result (Score: 0.3105): Neural networks are computing systems inspired by biological neural networks, consisting of intercon...


*Questions:**

Q- What happens when you query about topics not in your knowledge base?

-In your case:

Query: “Tell me about quantum computing”

It retrieved 1 document with a low score (0.3105).

Since quantum computing is not present in the knowledge base, the system tries to return the closest related topic. However, the similarity score is low, showing weak semantic relation.

Q- How does the threshold help filter irrelevant results?

Quantum computing → similarity 0.3105

Because threshold = 0.3, it allowed that result.

-The threshold ensures that only documents with sufficient similarity are returned. It prevents weakly related or unrelated documents from being retrieved, improving precision and reliability.

## Part 4: Optimizing the Retrieval Pipeline

### Understanding the Importance

As your knowledge base grows, retrieval performance becomes critical. You need efficient data structures and algorithms to handle thousands or millions of documents.

**Why this matters:** A retrieval system that works well with 10 documents might become unusable with 10,000. Learning optimization techniques early prevents scaling problems later.

In [22]:
### Task 4.1: Implement Batch Processing

#Create a function that efficiently handles multiple queries at once.
def batch_retrieve(queries, documents, embeddings, model, top_k=3):
    """
    Efficiently retrieve documents for multiple queries.

    Args:
        queries (list): List of query strings.
        documents (list): List of document texts.
        embeddings (numpy.ndarray): Pre-computed document embeddings.
        model (SentenceTransformer): Embedding model.
        top_k (int): Number of documents per query.

    Returns:
        dict: Mapping of query -> list of (document, similarity_score)
    """

    if not queries:
        raise ValueError("Query list cannot be empty.")

    results = {}

    # Step 1: Encode all queries at once (Batch Processing)
    query_embeddings = model.encode(queries, convert_to_numpy=True)

    # Step 2: Process each query
    for query, query_embedding in zip(queries, query_embeddings):

        similarities = []

        # Compare query with all document embeddings
        for doc, doc_embedding in zip(documents, embeddings):
            score = cosine_similarity(query_embedding, doc_embedding)
            similarities.append((doc, score))

        # Sort by similarity
        similarities.sort(key=lambda x: x[1], reverse=True)

        # Store top_k results
        results[query] = similarities[:top_k]

    return results

In [23]:
queries = [
    "What is deep learning?",
    "Explain neural networks",
    "How do computers learn?"
]

batch_results = batch_retrieve(queries, DOCUMENTS, embeddings, model, top_k=2)

for query, results in batch_results.items():
    print(f"\nQuery: {query}")
    for doc, score in results:
        print(f"Score: {score:.4f} | {doc[:80]}...")


Query: What is deep learning?
Score: 0.7955 | Deep learning is a type of machine learning based on artificial neural networks ...
Score: 0.5616 | Machine learning is a subset of artificial intelligence that enables systems to ...

Query: Explain neural networks
Score: 0.7010 | Neural networks are computing systems inspired by biological neural networks, co...
Score: 0.6182 | Deep learning is a type of machine learning based on artificial neural networks ...

Query: How do computers learn?
Score: 0.4925 | Machine learning is a subset of artificial intelligence that enables systems to ...
Score: 0.4567 | Supervised learning involves training a model on labeled data, where the correct...


In [24]:
## Task 4.2: Create a Retrieval Class

#Encapsulate your retrieval pipeline in a reusable class.
import numpy as np
from sentence_transformers import SentenceTransformer

class RetrievalPipeline:
    """
    A complete retrieval pipeline for RAG systems.
    """

    def __init__(self, model_name='all-MiniLM-L6-v2'):
        """Initialize the pipeline with an embedding model."""

        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None


    def add_documents(self, documents):
        """
        Add documents to the knowledge base.

        Args:
            documents (list): List of text documents.
        """

        if not documents:
            raise ValueError("Document list cannot be empty.")

        self.documents.extend(documents)

        # Create embeddings for all documents
        self.embeddings = self.model.encode(
            self.documents,
            convert_to_numpy=True
        )


    def retrieve(self, query, top_k=3, threshold=0.0):
        """
        Retrieve relevant documents for a query.

        Args:
            query (str): Search query.
            top_k (int): Number of results.
            threshold (float): Minimum similarity score.

        Returns:
            list: List of (document, score) tuples.
        """

        if not query:
            raise ValueError("Query cannot be empty.")

        if self.embeddings is None:
            raise ValueError("No documents added to the pipeline.")

        # Create query embedding
        query_embedding = self.model.encode(
            query,
            convert_to_numpy=True
        )

        results = []

        # Compute cosine similarity manually
        for doc, doc_embedding in zip(self.documents, self.embeddings):

            dot_product = np.dot(query_embedding, doc_embedding)
            norm_query = np.linalg.norm(query_embedding)
            norm_doc = np.linalg.norm(doc_embedding)

            if norm_query == 0 or norm_doc == 0:
                score = 0.0
            else:
                score = dot_product / (norm_query * norm_doc)

            if score >= threshold:
                results.append((doc, score))

        # Sort by similarity
        results.sort(key=lambda x: x[1], reverse=True)

        return results[:top_k]


    def get_stats(self):
        """Return statistics about the knowledge base."""

        return {
            "Total Documents": len(self.documents),
            "Embedding Shape": None if self.embeddings is None else self.embeddings.shape
        }

In [25]:
from documents import DOCUMENTS

pipeline = RetrievalPipeline()
pipeline.add_documents(DOCUMENTS)

print("Pipeline Statistics:")
print(pipeline.get_stats())

query = "What are neural networks?"
results = pipeline.retrieve(query, top_k=2)

print(f"\nQuery: {query}")
for doc, score in results:
    print(f"Score: {score:.4f} | {doc[:80]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline Statistics:
{'Total Documents': 10, 'Embedding Shape': (10, 384)}

Query: What are neural networks?
Score: 0.7993 | Neural networks are computing systems inspired by biological neural networks, co...
Score: 0.6365 | Deep learning is a type of machine learning based on artificial neural networks ...


## Part 5: Evaluation and Analysis

### Understanding the Importance

You can't improve what you don't measure. Evaluation metrics help you understand how well your retrieval system performs and guide optimization efforts.

**Why this matters:** Without proper evaluation, you won't know if changes to your pipeline improve or degrade performance. In production systems, this can mean the difference between helpful and harmful AI responses.

In [27]:
## Task 5.1: Implement Relevance Metrics

#Create functions to measure retrieval quality.
def calculate_precision_at_k(retrieved_docs, relevant_docs, k=3):
    """
    Calculate precision at k: proportion of retrieved docs that are relevant.

    Args:
        retrieved_docs (list): List of retrieved document indices.
        relevant_docs (set): Set of relevant document indices.
        k (int): Number of top results to consider.

    Returns:
        float: Precision score.
    """

    if k == 0:
        return 0.0

    # Consider only top-k retrieved documents
    top_k_docs = retrieved_docs[:k]

    # Count how many are relevant
    relevant_count = sum(1 for doc in top_k_docs if doc in relevant_docs)

    return relevant_count / k

In [28]:
def calculate_recall_at_k(retrieved_docs, relevant_docs, k=3):
    """
    Calculate recall at k: proportion of relevant docs that were retrieved.

    Args:
        retrieved_docs (list): List of retrieved document indices.
        relevant_docs (set): Set of relevant document indices.
        k (int): Number of top results to consider.

    Returns:
        float: Recall score.
    """

    if not relevant_docs:
        return 0.0

    # Consider only top-k retrieved documents
    top_k_docs = retrieved_docs[:k]

    # Count relevant retrieved documents
    relevant_count = sum(1 for doc in top_k_docs if doc in relevant_docs)

    return relevant_count / len(relevant_docs)

Precision@K:

Precision measures how many of the retrieved documents are actually relevant.

Recall@K:

Recall measures how many of the relevant documents were successfully retrieved.

In [29]:
### Task 5.2: Create Test Cases

#Define test queries with known relevant documents.
def evaluate_retrieval(pipeline, test_cases, k=3):
    """
    Evaluate retrieval performance on test cases.

    Args:
        pipeline (RetrievalPipeline): Retrieval pipeline instance.
        test_cases (list): List of (query, relevant_docs) tuples.
        k (int): Number of results to retrieve.

    Returns:
        dict: Dictionary with average precision and recall.
    """

    total_precision = 0
    total_recall = 0

    for query, relevant_docs in test_cases:

        # Step 1: Retrieve documents
        results = pipeline.retrieve(query, top_k=k)

        # Step 2: Get indices of retrieved documents
        retrieved_indices = []

        for doc, _ in results:
            index = pipeline.documents.index(doc)
            retrieved_indices.append(index)

        # Step 3: Calculate precision and recall
        precision = calculate_precision_at_k(
            retrieved_indices, relevant_docs, k
        )

        recall = calculate_recall_at_k(
            retrieved_indices, relevant_docs, k
        )

        total_precision += precision
        total_recall += recall

    # Step 4: Compute averages
    avg_precision = total_precision / len(test_cases)
    avg_recall = total_recall / len(test_cases)

    return {
        "Average Precision@K": round(avg_precision, 4),
        "Average Recall@K": round(avg_recall, 4)
    }

In [30]:
# Define test cases
test_cases = [
    ("What is machine learning?", {0, 7, 8}),
    ("Explain neural networks", {1, 6, 9}),
    ("How does computer vision work?", {3}),
]

# Evaluate
metrics = evaluate_retrieval(pipeline, test_cases, k=3)

print("Evaluation Results:")
print(metrics)

Evaluation Results:
{'Average Precision@K': 0.6667, 'Average Recall@K': 0.8889}


In [31]:
### Task 5.3: Analyze Retrieval Performance

#Run evaluation and analyze the results.
# Evaluate overall performance
metrics = evaluate_retrieval(pipeline, test_cases, k=3)

print("Overall Evaluation Results:")
print(metrics)


# Detailed per-query analysis
print("\nDetailed Per-Query Analysis:")

for query, relevant_docs in test_cases:

    results = pipeline.retrieve(query, top_k=3)

    retrieved_indices = [
        pipeline.documents.index(doc) for doc, _ in results
    ]

    precision = calculate_precision_at_k(retrieved_indices, relevant_docs, k=3)
    recall = calculate_recall_at_k(retrieved_indices, relevant_docs, k=3)

    print(f"\nQuery: {query}")
    print(f"Precision@3: {precision:.4f}")
    print(f"Recall@3: {recall:.4f}")


Overall Evaluation Results:
{'Average Precision@K': 0.6667, 'Average Recall@K': 0.8889}

Detailed Per-Query Analysis:

Query: What is machine learning?
Precision@3: 0.6667
Recall@3: 0.6667

Query: Explain neural networks
Precision@3: 1.0000
Recall@3: 1.0000

Query: How does computer vision work?
Precision@3: 0.3333
Recall@3: 1.0000


**Analysis Questions:**
- What is the average precision and recall of your system?

-The average precision is 0.6667 and recall is 0.8889.
This means the system retrieves most relevant documents, but sometimes includes a few extra ones.

- Which queries perform best/worst? Why?

-Best: “Explain neural networks” (Perfect precision and recall)

Worst: “How does computer vision work?” (Good recall but low precision because extra documents were retrieved)

- How could you improve retrieval quality?

-We can improve retrieval by increasing the similarity threshold, using a better embedding model, or refining chunking.

## Part 6: Real-World Application

### Understanding the Importance

This final section bridges the gap between learning and application. You'll work with a more realistic scenario that includes challenges you'll face in production systems.

**Why this matters:** Textbook examples rarely capture the messiness of real data. Learning to handle longer documents, mixed content types, and edge cases prepares you for actual RAG implementations.

In [32]:
### Task 6.1: Extended Knowledge Base

#Create a more complex dataset with longer documents.
EXTENDED_DOCS = [
    """
    Artificial Intelligence (AI) is transforming industries worldwide. Machine learning,
    a subset of AI, enables systems to learn from data without explicit programming.
    Companies use ML for predictive analytics, recommendation systems, and automation.
    The field has grown exponentially with the availability of big data and computational power.
    """,
    """
    Deep learning has revolutionized computer vision and natural language processing.
    Convolutional neural networks (CNNs) excel at image recognition tasks, achieving
    human-level performance in many cases. Applications include medical imaging,
    autonomous vehicles, and facial recognition systems.
    """,
    """
    Natural language processing enables machines to understand human language. Modern NLP
    uses transformer architectures like BERT and GPT. These models power chatbots,
    translation services, and search engines. NLP is crucial for human-computer interaction.
    """,
    # Add more documents as needed
]


In [33]:
pipeline_extended = RetrievalPipeline()
pipeline_extended.add_documents(EXTENDED_DOCS)

print(pipeline_extended.get_stats())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'Total Documents': 3, 'Embedding Shape': (3, 384)}


In [34]:
### Task 6.2: Handle Multi-Document Queries

#Implement a function that retrieves and combines information from multiple documents.
def multi_document_retrieve(query, pipeline, top_k=5, combine=True):
    """
    Retrieve and optionally combine information from multiple documents.

    Args:
        query (str): Search query.
        pipeline (RetrievalPipeline): RetrievalPipeline instance.
        top_k (int): Number of documents to retrieve.
        combine (bool): Whether to combine retrieved documents.

    Returns:
        str or list: Combined context string or list of (document, score).
    """

    if not query:
        raise ValueError("Query cannot be empty.")

    # Step 1: Retrieve documents
    results = pipeline.retrieve(query, top_k=top_k)

    if not results:
        return "No relevant documents found."

    # Step 2: Combine or return separately
    if combine:
        combined_text = ""

        for i, (doc, score) in enumerate(results, 1):
            combined_text += f"\n--- Document {i} (Score: {score:.4f}) ---\n"
            combined_text += doc.strip() + "\n"

        return combined_text.strip()

    else:
        return results

In [35]:
query = "How is AI used in real-world applications?"

combined_context = multi_document_retrieve(
    query,
    pipeline_extended,
    top_k=3,
    combine=True
)

print(combined_context)

--- Document 1 (Score: 0.5772) ---
Artificial Intelligence (AI) is transforming industries worldwide. Machine learning, 
    a subset of AI, enables systems to learn from data without explicit programming. 
    Companies use ML for predictive analytics, recommendation systems, and automation. 
    The field has grown exponentially with the availability of big data and computational power.

--- Document 2 (Score: 0.4083) ---
Natural language processing enables machines to understand human language. Modern NLP 
    uses transformer architectures like BERT and GPT. These models power chatbots, 
    translation services, and search engines. NLP is crucial for human-computer interaction.

--- Document 3 (Score: 0.3855) ---
Deep learning has revolutionized computer vision and natural language processing. 
    Convolutional neural networks (CNNs) excel at image recognition tasks, achieving 
    human-level performance in many cases. Applications include medical imaging, 
    autonomous vehicl

In [36]:
## Task 6.3: Build a Complete RAG Retrieval Demo

#Create a simple interactive demo of your retrieval system.
def interactive_retrieval_demo(pipeline):
    """
    Run an interactive demo of the retrieval system.
    """

    print("RAG Retrieval System Demo")
    print("=" * 50)
    print("Enter your queries (type 'quit' to exit)")
    print("Type 'stats' to view pipeline statistics")
    print("=" * 50)

    while True:

        query = input("\nEnter your query: ").strip()

        # Exit condition
        if query.lower() == "quit":
            print("\nExiting demo. Thank you!")
            break

        # Show statistics
        if query.lower() == "stats":
            print("\nPipeline Statistics:")
            print(pipeline.get_stats())
            continue

        # Empty input handling
        if not query:
            print("Please enter a valid query.")
            continue

        # Retrieve results
        results = pipeline.retrieve(query, top_k=3)

        if not results:
            print("No relevant documents found.")
            continue

        print("\nTop Retrieved Documents:")

        for i, (doc, score) in enumerate(results, 1):
            print(f"\nResult {i} | Score: {score:.4f}")
            print(doc[:200], "...")  # Show first 200 characters

In [38]:
while True:

    query = input("\nEnter your query: ").strip()

    # Exit option
    if query.lower() == "quit":
        print("Exiting demo. Thank you!")
        break

    # Show statistics
    if query.lower() == "stats":
        print("\nPipeline Statistics:")
        print(pipeline.get_stats())
        continue

    # Handle empty input
    if not query:
        print("Please enter a valid query.")
        continue

    # Retrieve documents
    results = pipeline.retrieve(query, top_k=3)

    if not results:
        print("No relevant documents found.")
        continue

    print("\nTop Retrieved Documents:")

    for i, (doc, score) in enumerate(results, 1):
        print(f"\nResult {i} | Score: {score:.4f}")
        print(doc[:200], "...")


Enter your query: What is machine learning?

Top Retrieved Documents:

Result 1 | Score: 0.8172
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. ...

Result 2 | Score: 0.5860
Supervised learning involves training a model on labeled data, where the correct output is known for each input example. ...

Result 3 | Score: 0.5767
Reinforcement learning is a machine learning approach where an agent learns to make decisions by performing actions and receiving rewards or penalties. ...

Enter your query: stats

Pipeline Statistics:
{'Total Documents': 10, 'Embedding Shape': (10, 384)}

Enter your query: quit
Exiting demo. Thank you!


## Challenge Task

### Challenge 1: Implement Re-ranking

After initial retrieval, re-rank results using a different strategy (e.g., keyword matching combined with semantic similarity).


In [39]:
import numpy as np

def rerank_with_keyword_boost(query, pipeline, top_k=3, alpha=0.8):
    """
    Re-rank retrieved documents using a combination of
    semantic similarity and keyword matching.

    Args:
        query (str): Search query.
        pipeline (RetrievalPipeline): Retrieval pipeline instance.
        top_k (int): Number of documents to return.
        alpha (float): Weight for semantic similarity (0–1).

    Returns:
        list: Re-ranked list of (document, final_score).
    """

    # Step 1: Get initial semantic retrieval (retrieve more for better re-ranking)
    initial_results = pipeline.retrieve(query, top_k=len(pipeline.documents))

    if not initial_results:
        return []

    query_words = set(query.lower().split())
    reranked_results = []

    for doc, semantic_score in initial_results:

        doc_words = set(doc.lower().split())

        # Step 2: Compute keyword overlap score
        keyword_overlap = len(query_words & doc_words)
        keyword_score = keyword_overlap / (len(query_words) + 1e-8)

        # Step 3: Combine scores using weighted sum
        final_score = alpha * semantic_score + (1 - alpha) * keyword_score

        reranked_results.append((doc, final_score))

    # Step 4: Sort by final combined score
    reranked_results.sort(key=lambda x: x[1], reverse=True)

    return reranked_results[:top_k]

In [40]:
query = "How do neural networks work?"

results = rerank_with_keyword_boost(query, pipeline, top_k=3)

for doc, score in results:
    print(f"\nScore: {score:.4f}")
    print(doc[:150], "...")


Score: 0.6552
Neural networks are computing systems inspired by biological neural networks, consisting of interconnected nodes that process information. ...

Score: 0.5599
Deep learning is a type of machine learning based on artificial neural networks with multiple layers, allowing computers to learn complex patterns. ...

Score: 0.3782
Transformers are a type of neural network architecture that uses self-attention mechanisms, revolutionary for NLP tasks. ...


### Challenge 2: Add Metadata Filtering

Extend your pipeline to support filtering documents by metadata (e.g., date, category, author).

In [41]:
DOCUMENTS_WITH_METADATA = [
    {
        "text": "Machine learning is a subset of artificial intelligence.",
        "category": "ML",
        "author": "Alice",
        "date": "2023-01-01"
    },
    {
        "text": "Deep learning uses neural networks with multiple layers.",
        "category": "DL",
        "author": "Bob",
        "date": "2023-02-15"
    },
    {
        "text": "Computer vision enables machines to interpret images.",
        "category": "CV",
        "author": "Alice",
        "date": "2023-03-10"
    }
]

In [42]:
import numpy as np
from sentence_transformers import SentenceTransformer

class RetrievalPipelineWithMetadata:

    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.documents = []          # stores full metadata objects
        self.embeddings = None

    def add_documents(self, documents):
        """
        documents: list of dictionaries with keys:
        'text', 'category', 'author', 'date'
        """
        self.documents.extend(documents)

        texts = [doc["text"] for doc in self.documents]
        self.embeddings = self.model.encode(texts, convert_to_numpy=True)

    def retrieve(self, query, top_k=3, category=None, author=None):

        query_embedding = self.model.encode(query, convert_to_numpy=True)

        results = []

        for doc, doc_embedding in zip(self.documents, self.embeddings):

            # --- Metadata filtering ---
            if category and doc["category"] != category:
                continue

            if author and doc["author"] != author:
                continue

            # --- Compute similarity ---
            score = np.dot(query_embedding, doc_embedding) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding)
            )

            results.append((doc, score))

        results.sort(key=lambda x: x[1], reverse=True)

        return results[:top_k]

In [43]:
pipeline_meta = RetrievalPipelineWithMetadata()
pipeline_meta.add_documents(DOCUMENTS_WITH_METADATA)

query = "How do machines learn?"

print("\nWithout Filter:")
results = pipeline_meta.retrieve(query, top_k=2)
for doc, score in results:
    print(f"{doc['text']} | Score: {score:.4f}")

print("\nFilter by Category = ML:")
results = pipeline_meta.retrieve(query, top_k=2, category="ML")
for doc, score in results:
    print(f"{doc['text']} | Score: {score:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Without Filter:
Machine learning is a subset of artificial intelligence. | Score: 0.5391
Computer vision enables machines to interpret images. | Score: 0.5053

Filter by Category = ML:
Machine learning is a subset of artificial intelligence. | Score: 0.5391


## Challenge 3: Implement Hybrid Search

Combine dense retrieval (embeddings) with sparse retrieval (keyword-based like BM25) for improved results.

In [44]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

class HybridRetrievalPipeline:

    def __init__(self, model_name='all-MiniLM-L6-v2'):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None
        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = None

    def add_documents(self, documents):
        self.documents = documents

        # Dense embeddings
        self.embeddings = self.model.encode(
            documents, convert_to_numpy=True
        )

        # Sparse TF-IDF representation
        self.tfidf_matrix = self.vectorizer.fit_transform(documents)

    def hybrid_retrieve(self, query, top_k=3, alpha=0.7):
        """
        alpha → weight for dense score
        (1 - alpha) → weight for sparse score
        """

        # ---- Dense Score ----
        query_embedding = self.model.encode(query, convert_to_numpy=True)

        dense_scores = []
        for doc_embedding in self.embeddings:
            score = np.dot(query_embedding, doc_embedding) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding)
            )
            dense_scores.append(score)

        dense_scores = np.array(dense_scores)

        # ---- Sparse Score (TF-IDF cosine similarity) ----
        query_tfidf = self.vectorizer.transform([query])

        sparse_scores = (self.tfidf_matrix @ query_tfidf.T).toarray().flatten()

        # ---- Normalize Scores ----
        dense_scores = dense_scores / (np.max(dense_scores) + 1e-8)
        sparse_scores = sparse_scores / (np.max(sparse_scores) + 1e-8)

        # ---- Combine ----
        hybrid_scores = alpha * dense_scores + (1 - alpha) * sparse_scores

        # ---- Rank ----
        ranked_indices = np.argsort(hybrid_scores)[::-1]

        results = []
        for idx in ranked_indices[:top_k]:
            results.append((self.documents[idx], hybrid_scores[idx]))

        return results

In [45]:
pipeline_hybrid = HybridRetrievalPipeline()
pipeline_hybrid.add_documents(DOCUMENTS)

query = "How do neural networks work?"

results = pipeline_hybrid.hybrid_retrieve(query, top_k=3)

for doc, score in results:
    print(f"\nScore: {score:.4f}")
    print(doc[:150], "...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Score: 1.0000
Neural networks are computing systems inspired by biological neural networks, consisting of interconnected nodes that process information. ...

Score: 0.7389
Deep learning is a type of machine learning based on artificial neural networks with multiple layers, allowing computers to learn complex patterns. ...

Score: 0.4831
Transformers are a type of neural network architecture that uses self-attention mechanisms, revolutionary for NLP tasks. ...


## Questions

1. What are the key trade-offs between chunk size and retrieval quality?

-If chunks are too small, the meaning may break and important context can be lost.
If chunks are too large, retrieval may become less precise because unrelated information gets mixed together.

So it’s always a balance between preserving context and maintaining precision.

2. How would you handle documents in multiple languages?

-I would use a multilingual embedding model that supports multiple languages.
Another option is translating all documents into a common language before creating embeddings.

The goal is to make sure semantic similarity works across languages.


3. What strategies could you use to update your knowledge base without recomputing all embeddings?

-Instead of recomputing everything, I would:

Generate embeddings only for new documents

Append them to the existing embedding store

Use a vector database for efficient updates

This saves time and computation.

4. How would you monitor retrieval quality in a production system?

-I would monitor:

Precision and recall metrics

User feedback

Query success rates

Low-confidence retrievals

Continuous evaluation helps ensure the system remains accurate over time.

5. What are the computational costs of your pipeline, and how would you optimize for large-scale deployment?

-The main costs come from:

Generating embeddings

Computing similarity across many documents

To optimize, I would:

Use vector databases

Use batch processing

Cache frequent queries

Use efficient indexing methods

This makes the system scalable for thousands or millions of documents.